# Lab 5 — NYSE Multi-Cluster View

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

## Learning objectives

1. Fit **K-Means** and **DBSCAN** on the same scaled symbol features.
2. Visualize both cluster assignments side-by-side (`avg_close` vs `volatility`).
3. Save `multi_cluster_view.png` to `output/`.
4. Explain why the two plots can disagree.

> **Checkpoints:** **25** symbols plotted · `multi_cluster_view.png` saved · K-Means & DBSCAN counts match Labs 1 & 3



## Why two plots?

We project **4-D** features onto 2-D (`avg_close` vs `volatility`) for visualization — clusters exist in full feature space.

| Panel | Method | Key visual cue |
|-------|--------|----------------|
| Left | K-Means k=4 | Every symbol colored; forced assignment |
| Right | DBSCAN | Label **-1** (noise) in sparse regions |

Symbols in sparse corners often become DBSCAN noise while K-Means still assigns them.

---

## 1. Load features and fit both clusterers

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-05":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "nyse" / "nyse_stocks.csv").is_file():
            GH_ROOT = parent
            break

OUTPUT_DIR = GH_ROOT / "hands-on" / "day-05" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

X_scaled = StandardScaler().fit_transform(features[FEATURE_COLUMNS])

kmeans_labels = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_scaled)
dbscan_labels = DBSCAN(eps=1.2, min_samples=3).fit_predict(X_scaled)

features = features.copy()
features["kmeans"] = kmeans_labels
features["dbscan"] = dbscan_labels

print(f"symbols plotted: {len(features)}")

---

## 2. Cluster counts

In [ ]:
kmeans_counts = dict(zip(*np.unique(kmeans_labels, return_counts=True)))
dbscan_counts = dict(zip(*np.unique(dbscan_labels, return_counts=True)))

print("Lab 5 — NYSE multi-cluster view")
print(f"K-Means cluster counts: {kmeans_counts}")
print(f"DBSCAN cluster counts: {dbscan_counts}")

---

## 3. Side-by-side scatter plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, labels, title in zip(
    axes,
    [kmeans_labels, dbscan_labels],
    ["K-Means (k=4)", "DBSCAN"],
    strict=True,
):
    scatter = ax.scatter(
        features["avg_close"],
        features["volatility"],
        c=labels,
        cmap="tab10",
        alpha=0.85,
    )
    ax.set_xlabel("avg_close")
    ax.set_ylabel("volatility")
    ax.set_title(title)
    fig.colorbar(scatter, ax=ax, label="cluster")

fig.tight_layout()
plot_path = OUTPUT_DIR / "multi_cluster_view.png"
fig.savefig(plot_path, dpi=100)
plt.show()

print(f"plot saved: {plot_path.name}")
print(f"full path: {plot_path}")

---

## 4. DBSCAN noise symbols (sparse regions)

In [ ]:
noise = features.loc[features["dbscan"] == -1, ["symbol", "avg_close", "volatility"]]
display(noise.sort_values("volatility").round(2))
print(f"noise count: {len(noise)}")

These symbols sit far from dense groups in scaled space — K-Means still assigns them to nearest centroids.

---

## 5. Where K-Means and DBSCAN disagree

In [ ]:
features["agree"] = features["kmeans"] == features["dbscan"]
disagree = features.loc[~features["agree"] | (features["dbscan"] == -1),
                      ["symbol", "kmeans", "dbscan"]]
display(disagree.sort_values("dbscan"))

---

## 6. Saved file preview

In [ ]:
if plot_path.is_file():
    display(Image(filename=str(plot_path)))
else:
    print("Run the plot cell above first.")

---

## 7. Checkpoint summary

In [ ]:
assert len(features) == 25
assert plot_path.is_file()
assert kmeans_counts == {0: 8, 1: 9, 2: 7, 3: 1}
assert dbscan_counts[-1] == 11
assert dbscan_counts[0] == 6 and dbscan_counts[1] == 4 and dbscan_counts[2] == 4
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why can two symbols look close on this 2-D plot but differ in 4-D cluster assignment?
2. Which method would you use for a mandatory segment label vs an optional watch list?
3. How would adding `avg_volume` to the axes change what you see?

**Previous:** [Lab 4 — Cluster metrics](lab04_cluster_metrics.ipynb)  
**Next:** [Lab 6 — Segmentation summary](lab06_segmentation_summary.ipynb)